# Task 1 训练结果概览

本 Notebook 展示 Q-learning 与 DQN 在 Task 1（coin navigation）的六条正式训练链：每种算法使用 seed 11、22、33，各 500 局。统计重复单位是 **training seed**，不是 1,500 个 round。图表文字使用英文，所有统计和绘图由 `experiments.analyze_training` 的公共接口完成。

In [1]:
from pathlib import Path
import csv
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'experiments').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.analyze_training import analyze_training_runs

RUNS = [
    ROOT / 'runs/formal_q_v1_r1_s11_t1_r500',
    ROOT / 'runs/formal_q_v1_r1_s22_t1_r500',
    ROOT / 'runs/formal_q_v1_r1_s33_t1_r500',
    ROOT / 'runs/formal_dqn_v1_r1_s11_t1_r500',
    ROOT / 'runs/formal_dqn_v1_r1_s22_t1_r500',
    ROOT / 'runs/formal_dqn_v1_r1_s33_t1_r500',
]
OUTPUT = ROOT / 'results/task1_analysis'
report = analyze_training_runs(RUNS, OUTPUT, rolling_window=25)
print(f'Generated {len(report["round_metrics"])} round records in {OUTPUT.relative_to(ROOT)}')

Generated 3000 round records in results/task1_analysis


## 数据完整性与每个 seed 的汇总

下表来自分析器写出的 CSV。Task 1 中 score 与 coins 相同，且当前场景的 kills、suicides、crates、bombs、invalid 和 deaths 都为零，因此没有为这些常量另画图。coins 仅在每个 round 结束时记录，不能解释为局内逐 step 的获取过程。

In [2]:
from html import escape
from IPython.display import HTML, display

def show_csv(filename, columns=None):
    with (OUTPUT / filename).open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    columns = columns or list(rows[0])
    header = ''.join(f'<th>{escape(column)}</th>' for column in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(row[column]))}</td>' for column in columns) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))

show_csv('run_summary.csv', ['algorithm', 'seed', 'rounds', 'action_steps', 'mean_coins', 'first_100_mean_coins', 'last_100_mean_coins', 'coins_change_last_minus_first_100', 'final_q_states', 'final_dqn_last_update_loss', 'final_updates'])
show_csv('algorithm_summary.csv')

algorithm,seed,rounds,action_steps,mean_coins,first_100_mean_coins,last_100_mean_coins,coins_change_last_minus_first_100,final_q_states,final_dqn_last_update_loss,final_updates
q_learning,11,500,200000,24.318,21.04,27.2,6.16,50,,
q_learning,22,500,200000,23.752,21.45,25.85,4.400000000000002,50,,
q_learning,33,500,200000,23.96,21.3,26.7,5.399999999999999,50,,
dqn,11,500,200000,24.764,20.95,28.04,7.09,,0.11644449084997177,198001
dqn,22,500,200000,24.622,21.18,27.92,6.740000000000002,,0.02398815006017685,198001
dqn,33,500,200000,24.572,21.43,27.12,5.690000000000001,,0.04081317409873009,198001


algorithm,n_seeds,mean_coins,mean_coins_sample_sd,first_100_mean_coins,first_100_mean_coins_sample_sd,last_100_mean_coins,last_100_mean_coins_sample_sd,coins_change_last_minus_first_100,coins_change_last_minus_first_100_sample_sd,mean_reward,mean_reward_sample_sd
dqn,3,24.652666666666665,0.09960589005341668,21.186666666666667,0.2400694344004114,27.693333333333335,0.5001333155602943,6.506666666666668,0.728583099813146,20.652666666666537,0.09960589005340872
q_learning,3,24.01,0.2862935556382656,21.263333333333332,0.20744477176668835,26.583333333333332,0.6825198409814411,5.32,0.8827230596285554,20.04333333333322,0.2847197452466713


## 核心图表

每条淡线是一个 seed 的每局 coins；粗线为包含当前局的 trailing 25-round mean。算法比较图中的阴影是 seed 间的 sample SD，不是置信区间。

![Q-learning coins](../results/task1_analysis/coins_per_round_q_learning.png)

![DQN coins](../results/task1_analysis/coins_per_round_dqn.png)

![Algorithm comparison](../results/task1_analysis/coins_algorithm_comparison.png)

![First versus last 100 rounds](../results/task1_analysis/coins_first_vs_last100.png)

## 诊断与当前观察

其余诊断图保存在 `results/task1_analysis/`：`cumulative_coins.png`、`reward_epsilon_progress.png` 与 `learner_diagnostics.png`。分析器会同时验证六条 run 的 exploration schedule 是否相同，并把结果写入 `analysis_metadata.json`。

可由上表和图表支持的观察仅限于：每个算法、每个 seed 的全程与前/末 100 局 mean coins，以及跨 seed 的均值和离散度。它们适合诊断学习过程，**不自动推出**奖励、epsilon 或网络超参数应如何修改。Task 2 的进入决定仍需独立的冻结 stage-gate evaluation；当前尚未在本报告中提供该评估数据。